# RAG Pipeline Test
Dieses Notebook testet alle Komponenten der RAG-Pipeline Schritt für Schritt.

## 0. Setup — Pfade & Imports

In [1]:
import sys
sys.path.append('../')  # damit Python die Module im Hauptordner findet

from dotenv import load_dotenv
load_dotenv('../.env')  # API-Keys laden

True

## 1. Loader testen

In [2]:
from rag.loader import DocumentLoader

# Test-PDF laden (liegt im selben Ordner wie dieses Notebook)
loader = DocumentLoader('machine_learning_intro.pdf')
documents = loader.load()

print(f'Anzahl Seiten: {len(documents)}')
print(f'Erste Seite (erste 300 Zeichen):')
print(documents[0].page_content[:300])
print(f'Metadaten: {documents[0].metadata}')

Anzahl Seiten: 3
Erste Seite (erste 300 Zeichen):
Introduction to Machine Learning
Chapter 1: What is Machine Learning?
Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs that can
access data and u
Metadaten: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-03-06T17:28:33+00:00', 'source': 'machine_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 2. Chunker testen

In [3]:
from rag.chunking import DocumentChunker

chunker = DocumentChunker(chunk_size=1000, chunk_overlap=200)
chunks = chunker.chunk(documents)

print(f'Anzahl Chunks: {len(chunks)}')
# print(f'Erster Chunk (erste 300 Zeichen):')
# print(chunks[0].page_content[:300])
print(f'Metadaten: {chunks[0].metadata}')

Anzahl Chunks: 7
Metadaten: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-03-06T17:28:33+00:00', 'source': 'machine_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 3. Embeddings testen

In [4]:
from rag.embeddings import EmbeddingGenerator

embedding_gen = EmbeddingGenerator()
embedding_model = embedding_gen.get_embedding_model()

# Einen Test-Vektor generieren
test_vector = embedding_model.embed_query('Was ist maschinelles Lernen?')

print(f'Vektor-Dimension: {len(test_vector)}')
print(f'Erste 5 Werte: {test_vector[:5]}')

Vektor-Dimension: 3072
Erste 5 Werte: [-0.024391094, 0.011332491, 0.0010179622, -0.062313493, -0.015748147]


## 4. VectorStore testen

In [ ]:
from rag.vectorstore import VectorStore

# VectorStore initialisieren (speichert in ./vector_store)
vs = VectorStore()

# Chunks speichern
vs.store(chunks)
print('Chunks erfolgreich gespeichert!')

# Retriever holen und testen
strategies = ['similarity', 'mmr', 'multi_query']
for strategy in strategies:
    retriever = vs.get_retriever(strategy=strategy, k=3)
    results = retriever.invoke('What is machine learning?')
    
    print(f'\n=== Retrieval Strategy: {strategy} ===')
    print(f'Gefundene Chunks: {len(results)}')
    for i, doc in enumerate(results):
        print(f'\n--- Chunk {i+1} (Seite {doc.metadata.get("page", 0) + 1}) ---')
        print(doc.page_content[:200])


print(f'Gefundene Chunks: {len(results)}')
for i, doc in enumerate(results):
    print(f'\n--- Chunk {i+1} (Seite {doc.metadata.get("page", 0) + 1}) ---')
    print(doc.page_content[:200])

Chunks erfolgreich gespeichert!
Gefundene Chunks: 3

--- Chunk 1 (Seite 1) ---
Introduction to Machine Learning
Chapter 1: What is Machine Learning?
Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve
from experience withou

--- Chunk 2 (Seite 1) ---
Introduction to Machine Learning
Chapter 1: What is Machine Learning?
Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve
from experience withou

--- Chunk 3 (Seite 1) ---
Introduction to Machine Learning
Chapter 1: What is Machine Learning?
Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve
from experience withou


## 4b. Retriever-Strategien einzeln testen

In [ ]:
# Jede Strategie einzeln testen — mit Assertion und sauberem Output
test_query = "What is machine learning?"
strategies = ["similarity", "mmr", "multi_query"]

for strategy in strategies:
    retriever = vs.get_retriever(strategy=strategy, k=3)
    results = retriever.invoke(test_query)

    assert len(results) > 0, f"Strategie '{strategy}' hat keine Ergebnisse zurückgegeben!"

    print(f"\n{'='*50}")
    print(f"Strategie: {strategy}")
    print(f"Anzahl Chunks: {len(results)}")
    for i, doc in enumerate(results):
        print(f"  Chunk {i+1} | Seite {doc.metadata.get('page', 0) + 1} | {doc.page_content[:120]}...")

print("\nAlle Strategien erfolgreich getestet!")

## 5. RAG Chain testen

In [6]:
from rag.chain import RAGChain

# Mit Groq testen
chain = RAGChain(retriever=retriever, llm='groq')
result = chain.run('Was ist maschinelles Lernen?')

print('=== ANTWORT ===')
print(result['answer'])

print('\n=== QUELLEN ===')
for doc in result['sources']:
    print(f"- Seite {doc.metadata.get('page', 0) + 1}: {doc.page_content[:100]}...")

=== ANTWORT ===
Maschinelles Lernen (ML) ist ein Teilbereich der künstlichen Intelligenz (KI), der es Systemen ermöglicht, aus Erfahrungen zu lernen und sich zu verbessern, ohne explizit programmiert zu werden. Es konzentriert sich auf die Entwicklung von Computerprogrammen, die auf Daten zugreifen und diese verwenden können, um selbst zu lernen.

Der Prozess des maschinellen Lernens beginnt mit Beobachtungen oder Daten, wie z.B. Beispielen, direkter Erfahrung oder Anweisungen. Das primäre Ziel ist es, Computern zu ermöglichen, automatisch ohne menschliche Eingriffe zu lernen und ihre Aktionen entsprechend anzupassen.

Es gibt drei Haupttypen des maschinellen Lernens:

1. Überwachtes Lernen: Das Algorithmus wird mit beschrifteten Daten trainiert. Das Modell lernt, Eingabedaten auf die korrekte Ausgabe abzubilden, basierend auf Beispiel-Eingabe-Ausgabe-Paaren. Häufige Beispiele sind lineare Regression, Entscheidungsbäume und neuronale Netze.
2. Unüberwachtes Lernen: Der Algorithmus wird

In [7]:
# Mit OpenRouter testen
chain_openrouter = RAGChain(retriever=retriever, llm='openrouter')
result_openrouter = chain_openrouter.run('Was ist maschinelles Lernen?')

print('=== ANTWORT (OpenRouter) ===')
print(result_openrouter['answer'])

=== ANTWORT (OpenRouter) ===
Maschinelles Lernen (Machine Learning, ML) ist ein Teilbereich der Künstlichen Intelligenz (KI), der es Systemen ermöglicht, aus Erfahrungen zu lernen und sich zu verbessern, ohne explizit programmiert zu werden. Es entwickelt Algorithmen, die auf Daten zugreifen, daraus Muster erkennen und eigenständig Vorhersagen oder Entscheidungen treffen können. Der Prozess beginnt mit Beobachtungen oder Daten (z. B. Beispielen oder direkter Erfahrung), mit dem Ziel, dass Computer automatisch lernen und ihre Aktionen anpassen – ohne menschliches Eingreifen.
